# Generative Adversarial Networks

**Generative Adversarial Network** (GAN) ka main goal aisi images banana hai jo training dataset ki images jaisi hon (lekin bilkul same na hon).

GAN mein do neural networks hote hain, jinhe ek-dusre ke against train kiya jaata hai:

* **Generator** ek random vector leta hai aur usse ek image generate karni hoti hai.
* **Discriminator** ek aisa network hai jise original image (training dataset se) aur generator dwara generate ki gayi image ke beech fark pehchanna hota hai.

In [ ]:
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
import matplotlib.pyplot as plt
import numpy as np

## Generator

Generator ka kaam ek certain size ka random vector (yeh autoencoders ke latent vector jaisa hi hai) lena aur target image generate karna hai. Yeh autoencoder ke generative side se bahut milta-julta hai.

Hamare example mein, hum dense neural network aur MNIST dataset use karenge.

In [ ]:
generator = Sequential()
generator.add(Dense(256, input_shape=(100,)))
generator.add(LeakyReLU(alpha=0.2))
generator.add(BatchNormalization(momentum=0.8))
generator.add(Dense(512))
generator.add(LeakyReLU(alpha=0.2))
generator.add(BatchNormalization(momentum=0.8))
generator.add(Dense(1024))
generator.add(LeakyReLU(alpha=0.2))
generator.add(BatchNormalization(momentum=0.8))
generator.add(Dense(784, activation='tanh'))
generator.add(Reshape((28,28)))

optimizer = keras.optimizers.Adam(lr=0.0002, decay=8e-9)

generator.compile(loss='binary_crossentropy',optimizer=optimizer,metrics=['accuracy'])


Generator mein use kiye gaye kuch tricks:
* ReLU ki jagah hum **Leaky ReLU** use karte hain, yaani ek aisa ReLU jo negative $x$ ke liye bilkul 0 nahi hota, balki ek aur linear function hota hai jiski slope bahut chhoti hoti hai. Yeh important hai kyunki yeh gradient descent ko values badhane mein madad karta hai, chahe hum ReLU ke negative side par hon (jahan values 0 hoti hain).
* Training ko stable banane ke liye hum Batch Normalization use karte hain.
* Last layer par activation function `tanh` hota hai, jisse output [-1,1] ki range mein rehta hai.

## Discriminator

Discriminator ek traditional image classification network hai. Hamare pehle example mein, hum dense classifier hi use karenge.

In [ ]:
discriminator = Sequential()
discriminator.add(Flatten(input_shape=(28,28)))
discriminator.add(Dense(784))

discriminator.add(LeakyReLU(alpha=0.2))
discriminator.add(Dense(784//2))
discriminator.add(LeakyReLU(alpha=0.2))

discriminator.add(Dense(1, activation='sigmoid'))

discriminator.compile(loss='binary_crossentropy',optimizer=optimizer,metrics=['accuracy'])

Hum ek adversarial network bhi define karenge, jo generator ke baad discriminator hota hai. Yeh network ek noise vector se shuru hota hai, aur ek binary result return karta hai.

In [ ]:
discriminator.trainable = False
adversarial = Sequential()
adversarial.add(generator)
adversarial.add(discriminator)
adversarial.compile(loss='binary_crossentropy', optimizer=optimizer)

## Dataset load karna

Hum MNIST dataset use karenge, jo Keras mein shamil hai:

In [ ]:
(X_train, _), (_, _) = keras.datasets.mnist.load_data()
X_train = (X_train.astype(np.float32) - 127.5) / 127.5

## Network training

Training ke har step mein, humare paas do phases hote hain:

* Discriminator ki training:
   - Hum kuch random vectors `noise` generate karte hain. Training minibatch mein hoti hai, isliye hum `batch//2` vectors use karke `batch//2` generated images banate hain
   - Dataset se `batch//2` random images sample karte hain
   - Discriminator ko 50% real aur 50% generated images par train karte hain, aur corresponding labels (0 ya 1) dete hain
 * Combined adversarial model se generator ko train karte hain, random vectors ko input ke roop mein pass karte hain, aur output mein 1 expect karte hain (jo real images ke barabar hai)

In [ ]:
def plotn(n):
  noise = np.random.normal(0, 1, (n,100))
  imgs = generator.predict(noise)
  fig,ax = plt.subplots(1,n)
  for i,im in enumerate(imgs):
    ax[i].imshow(im.reshape(28,28))
  plt.show()

In [ ]:
batch=32
for cnt in range(3000):
## train discriminator
  random_index =  np.random.randint(0, len(X_train) - batch//2)
  legit_images = X_train[random_index : random_index + batch//2].reshape(batch//2, 28, 28)
  gen_noise = np.random.normal(0, 1, (batch//2,100))
  syntetic_images = generator.predict(gen_noise)
  x_combined_batch = np.concatenate((legit_images, syntetic_images))
  y_combined_batch = np.concatenate((np.ones((batch//2, 1)), np.zeros((batch//2, 1))))
  d_loss = discriminator.train_on_batch(x_combined_batch, y_combined_batch)
# train generator
  noise = np.random.normal(0, 1, (batch,100))
  y_mislabled = np.ones((batch, 1))
  g_loss = adversarial.train_on_batch(noise, y_mislabled)
  if cnt%500==0:
    print ('epoch: %d, [Discriminator :: d_loss: %f], [ Generator :: loss: %f]' % (cnt, d_loss[0], g_loss))
    plotn(5)

> **Task**: Aap is GAN ko poore MNIST dataset par train kar sakte hain aur dekh sakte hain ki yeh kitna achha ban sakta hai

## DCGAN

Pichhle example mein, humne generator aur discriminator, dono ke liye dense networks use kiye the, lekin humein pata hai ki images ke saath kaam karte waqt CNNs behtar performance dete hain. **Deep Convolutional GAN** upar diye gaye architecture jaisa hi hai, lekin yeh generator aur discriminator ke liye convolutional layers use karta hai.

Yahan main challenge generator ke liye ek architecture banana hai, kyunki ise traditional CNN ke ulat kaam karna hota hai - ise feature vector se image generate karni hoti hai. Ek tarah se, yeh autoencoders ke decoder part jaisa hai. Isi wajah se hum generator mein `Conv2DTranspose` layers use karenge.

In [ ]:
(X_train, _), (_, _) = keras.datasets.mnist.load_data()
X_train = (X_train.astype(np.float32)-127.5) / 127.5
print(X_train.min(),X_train.max())

In [ ]:
generator = Sequential()
generator.add(Dense(128 * 7 * 7, activation="relu", input_dim=100))
generator.add(Reshape((7, 7, 128)))
generator.add(UpSampling2D())
generator.add(Conv2DTranspose(128, kernel_size=3, padding="same"))
generator.add(BatchNormalization(momentum=0.8))
generator.add(Activation("relu"))
generator.add(UpSampling2D())
generator.add(Conv2DTranspose(64, kernel_size=3, padding="same"))
generator.add(BatchNormalization(momentum=0.8))
generator.add(Activation("relu"))
generator.add(Conv2DTranspose(1, kernel_size=3, padding="same"))
generator.add(Activation("tanh"))

optimizer = keras.optimizers.Adam(0.0001) #, 0.5)

generator.compile(loss='binary_crossentropy',optimizer=optimizer,metrics=['accuracy'])
generator.summary()

In [ ]:
discriminator = Sequential()

discriminator.add(Conv2D(32, kernel_size=3, strides=2, input_shape=(28,28,1), padding="same"))
discriminator.add(LeakyReLU(alpha=0.2))
discriminator.add(Dropout(0.25))
discriminator.add(Conv2D(64, kernel_size=3, strides=2, padding="same"))
discriminator.add(ZeroPadding2D(padding=((0,1),(0,1))))
discriminator.add(BatchNormalization(momentum=0.8))
discriminator.add(LeakyReLU(alpha=0.2))
discriminator.add(Dropout(0.25))
discriminator.add(Conv2D(128, kernel_size=3, strides=2, padding="same"))
discriminator.add(BatchNormalization(momentum=0.8))
discriminator.add(LeakyReLU(alpha=0.2))
discriminator.add(Dropout(0.25))
discriminator.add(Conv2D(256, kernel_size=3, strides=1, padding="same"))
discriminator.add(BatchNormalization(momentum=0.8))
discriminator.add(LeakyReLU(alpha=0.2))
discriminator.add(Dropout(0.25))
discriminator.add(Flatten())
discriminator.add(Dense(1, activation='sigmoid'))

discriminator.compile(loss='binary_crossentropy',optimizer=optimizer)

In [ ]:
discriminator.trainable = False
adversarial = Sequential()
adversarial.add(generator)
adversarial.add(discriminator)
adversarial.compile(loss='binary_crossentropy', optimizer=optimizer)

In [ ]:
batch=32
y_labeled = np.ones((batch, 1))
y_mislabeled = np.zeros((batch, 1))
for cnt in range(1000):
## train discriminator
  random_index =  np.random.randint(0, len(X_train) - batch)
  legit_images = X_train[random_index : random_index + batch].reshape(batch, 28, 28, 1)
  gen_noise = np.random.normal(0, 1, (batch,100))
  syntetic_images = generator.predict(gen_noise)
  d_loss_1 = discriminator.train_on_batch(legit_images, y_labeled)
  d_loss_2 = discriminator.train_on_batch(syntetic_images, y_mislabeled)
  d_loss = 0.5*np.add(d_loss_1,d_loss_2)
# train generator
  g_loss = adversarial.train_on_batch(gen_noise, y_labeled)
  if cnt%100==0:
    print ('epoch: %d, [Discriminator :: d_loss: %f], [ Generator :: loss: %f]' % (cnt, d_loss, g_loss))
    plotn(5)

> **Task**: DCGAN se zyada complex colored images banane ki koshish kijiye - jaise, [CIFAR-10](https://keras.io/api/datasets/cifar10/) dataset se ek class lijiye.

## Paintings par training

GAN training ke liye ek achha candidate human artists dwara banayi gayi paintings hain. Neeche ek sample image hai jo DCGAN ne [WikiArt](http://wikiart.org) ke dataset par train hokar banayi hai. Is image ko [Azure Machine Learning ka use karte hue](https://soshnikov.com/scienceart/creating-generative-art-using-gan-on-azureml/) [KeraGAN](http://github.com/shwars/keragan) library se taiyaar kiya gaya hai.

![](https://soshnikov.com/images/artartificial/Flo1.jpg)

(Photo [Art of Artificial](https://soshnikov.com/museum/art-artificial/) collection se)

## References

* [Keras mein alag-alag toy GAN architectures ka implementation](https://github.com/eriklindernoren/Keras-GAN)
* [KeraGAN library](http://github.com/shwars/Keragan)
* [Azure ML par GANs banane ke baare mein blog post](https://soshnikov.com/scienceart/creating-generative-art-using-gan-on-azureml-ru/)

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->